<a href="https://colab.research.google.com/github/mateocgonzalez07/Proyecto-final-Ciencias-planetarias/blob/main/HARPS_LHS1140.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Descarga de HARPS para LHS 1140 (TOI-256)

Basado en el notebook de JuanDa (secciones 1, 2, 4, 5, 7 y 10.1 a 10.3). La sección 10.4 es la parte que falta y la escribo yo.

Objetivo: bajar los `.tar` de HARPS, sacar el `e2ds_A` de cada uno y dejarlo en la estructura que pide LBL.

## 0. Instalación y librerías

In [ ]:
%pip -Uq install pyvo

In [ ]:
from astropy.coordinates import SkyCoord
from astropy.io import fits
from astropy.table import Table
import pyvo
import numpy as np
import requests
from pathlib import Path
import tarfile
import io
from astropy.io.fits import open as fits_open

## 1. Coordenadas de la estrella

`from_name` le pregunta a Sesame (CDS) dónde está el objeto. ESO pide las coordenadas en grados.

In [ ]:
LHS_1140_coord = SkyCoord.from_name('LHS 1140')
ra = LHS_1140_coord.ra.deg
dec = LHS_1140_coord.dec.deg
print(f'RA (grados): {ra}')
print(f'Dec (grados): {dec}')

RA (grados): 11.24721214
Dec (grados): -15.27153968


## 2. Conexión al archivo de ESO

TAP es la ventanilla para hacerle preguntas al catálogo y ADQL es el idioma de las preguntas. Crear `tap` todavía no pregunta nada.

In [ ]:
tap = pyvo.dal.TAPService('https://archive.eso.org/tap_obs')
print(tap)

TAPService(baseurl : 'https://archive.eso.org/tap_obs', description : 'None')


## 3. Observaciones cerca de la estrella

Se busca por posición (círculo de 1 arcmin) y no por nombre, porque cada observador escribe el nombre distinto. Los nombres de columna (`s_ra`, `s_dec`) van sin `{}` y las variables de Python (`{ra}`, `{dec}`, `{radio}`) van con `{}`.

In [ ]:
radio = 1/60   # 1 arcmin, en grados

pregunta = f'''
SELECT dp_id, instrument_name, target_name, t_min, dataproduct_type, snr,
access_url
FROM ivoa.ObsCore
WHERE CONTAINS(POINT('ICRS', s_ra, s_dec),
               CIRCLE('ICRS', {ra}, {dec}, {radio})) = 1
'''

obs = tap.search(pregunta).to_table()
print(f'Productos encontrados: {len(obs)}')

Productos encontrados: 423


## 4. Solo HARPS

Máscara de True/False: `&` es "y" y cada condición va entre paréntesis. Deben salir 296.

In [ ]:
HARPS = obs[(obs['instrument_name'] == 'HARPS') & (obs['dataproduct_type'] == 'spectrum')]
HARPS.sort('t_min')
print(f'Espectros de HARPS: {len(HARPS)}')

Espectros de HARPS: 296


## 5. Función de descarga

- Si el archivo ya existe, no lo baja otra vez.
- Descarga de a 1 MB (`stream=True`) para no llenar la RAM.
- `raise_for_status()` frena si el servidor responde con error.
- Escribe en `.part` y renombra al final, así un archivo final siempre está completo.

In [ ]:
def descargar(url, destino):
    destino = Path(destino)
    if destino.exists():
        print(f'Ya existe, no lo bajo otra vez: {destino.name}')
        return destino
    destino.parent.mkdir(parents=True, exist_ok=True)
    temporal = destino.with_name(destino.name + '.part')

    with requests.get(url, stream=True, timeout=120) as r:
        r.raise_for_status()
        with open(temporal, 'wb') as f:
            for pedazo in r.iter_content(chunk_size=1024*1024):   # 1 MB
                f.write(pedazo)

    temporal.rename(destino)
    print(f'Descargado: {destino.name}')
    return destino

## 6. Archivos de un producto de HARPS

Cada producto tiene solo 2 archivos: el espectro 1D (`SCIENCE.SPECTRUM`) y un `.tar` (`ANCILLARY.HARPSTAR`) con los productos intermedios. El que sirve para LBL es el `.tar`.

In [ ]:
un_producto_harps = HARPS['dp_id'][0]
print(f'Producto: {un_producto_harps}')

archivos_harps = tap.search(f'''
SELECT archive_id, eso_category, original_filename, access_url, access_estsize
FROM phase3v2.product_files
WHERE product_id = '{un_producto_harps}'
''').to_table()

archivos_harps['eso_category', 'original_filename', 'access_estsize'].pprint_all()

Producto: ADP.2015-11-24T02:01:17.473
   eso_category                          original_filename                       access_estsize
                                                                                     kbyte     
------------------ ------------------------------------------------------------- --------------
  SCIENCE.SPECTRUM HARPS.2015-11-23T02:59:06.381_s1d_A_DRS_HARPS_3.8_ESOSDP.fits           5264
ANCILLARY.HARPSTAR               HARPS.2015-11-23T02:59:06.381_DRS_HARPS_3.8.tar           6973


## 7. Descargar y abrir un `.tar`

`tarfile.open` abre la caja sin sacar nada, `.getmembers()` lista lo que hay adentro (`.name`, `.size`) y `.extractfile()` lee un archivo sin guardarlo en disco.

Debe haber 11 archivos. El que usa LBL es `..._e2ds_A.fits` (4096 píxeles × 72 órdenes). El blaze **no** viene en el `.tar`.

In [ ]:
es_tar = [str(c) == 'ANCILLARY.HARPSTAR' for c in archivos_harps['eso_category']]
fila_tar = archivos_harps[es_tar][0]

ruta_tar = descargar(fila_tar['access_url'],
                     Path('datos_LHS_1140_TAR') / 'HARPS' / f"{fila_tar['archive_id']}.tar")

with tarfile.open(ruta_tar) as caja:
    for miembro in caja.getmembers():
        print(f'{miembro.size/1e6:8.2f} MB   {miembro.name}')

Ya existe, no lo bajo otra vez: ADP.2015-11-24T02:01:17.500.tar
    0.04 MB   data/reduced/2015-11-22/HARPS.2015-11-23T02:59:06.381_INT_GUIDE.fits
    0.25 MB   data/reduced/2015-11-22/HARPS.2015-11-23T02:59:06.381_bis_M2_A.fits
    0.25 MB   data/reduced/2015-11-22/HARPS.2015-11-23T02:59:06.381_bis_M2_B.fits
    0.30 MB   data/reduced/2015-11-22/HARPS.2015-11-23T02:59:06.381_ccf_M2_A.fits
    0.00 MB   data/reduced/2015-11-22/HARPS.2015-11-23T02:59:06.381_ccf_M2_A.tbl
    0.29 MB   data/reduced/2015-11-22/HARPS.2015-11-23T02:59:06.381_ccf_M2_B.fits
    0.00 MB   data/reduced/2015-11-22/HARPS.2015-11-23T02:59:06.381_ccf_M2_B.tbl
    1.43 MB   data/reduced/2015-11-22/HARPS.2015-11-23T02:59:06.381_e2ds_A.fits
    1.41 MB   data/reduced/2015-11-22/HARPS.2015-11-23T02:59:06.381_e2ds_B.fits
    1.50 MB   data/reduced/2015-11-22/HARPS.2015-11-23T02:59:06.381_s1d_A.fits
    1.50 MB   data/reduced/2015-11-22/HARPS.2015-11-23T02:59:06.381_s1d_B.fits


## 8. ¿Qué blaze usó este espectro?

El pipeline anota en el header del `e2ds` qué calibraciones usó. `extractfile(...).read()` da los bytes y `io.BytesIO` los envuelve para que `fits.open` los trate como un archivo.

In [ ]:
with tarfile.open(ruta_tar) as caja:
    m = [x for x in caja.getmembers() if x.name.endswith('e2ds_A.fits')][0]
    datos = caja.extractfile(m).read()

with fits.open(io.BytesIO(datos)) as hdul:
    hdul.info()
    h = hdul[0].header
    for clave in h:
        if any(s in clave.upper() for s in ['BLAZE', 'FLAT FILE', 'TH FILE']):
            print(clave, '=', h[clave])

Filename: <class '_io.BytesIO'>
No.    Name      Ver    Type      Cards   Dimensions   Format
  0  PRIMARY       1 PrimaryHDU    3059   (4096, 72)   float32   
ESO DRS CAL FLAT FILE = HARPS.2015-11-22T20:22:37.879_flat_A.fits
ESO DRS BLAZE FILE = HARPS.2015-11-22T20:22:37.879_blaze_A.fits
ESO DRS CAL TH FILE = HARPS.2015-11-22T20:37:37.636_wave_A.fits
ESO DRS CAL TH FILE LINES = line_list_HARPS_A.dat


---
# 9. Mi parte (paso 10.4 del notebook de JuanDa)

Pendiente, empezando con 3 productos de prueba:

1. Ciclo `for` sobre `HARPS`: consultar `product_files` de cada `dp_id`, quedarme con la fila `ANCILLARY.HARPSTAR` y descargar el `.tar` (~7 MB cada uno, ~2 GB en total).
2. Sacar solo el `e2ds_A` de cada `.tar` y guardarlo.
3. Leer `ESO DRS BLAZE FILE` de cada header y armar la lista de blazes distintos (para comparar con lo que pase Camilo).
4. Organizar en `science/LHS1140/` y `calib/` (estructura de LBL).
5. Guardar un manifiesto CSV para saber qué falta si se corta.

Ojo: en Colab todo se borra al cerrar la sesión, así que conviene guardar en Google Drive.

Pregunta abierta: ¿LBL necesita también el `wave_A` del header o lo lee del propio `e2ds`? Revisar la documentación de LBL (formato HARPS ORIG).

## Paso 1: ciclo de prueba con 3 productos


In [ ]:
for row in HARPS:
    # 1. Consultar los archivos de esta observación
    archivos_harps = tap.search(f'''
    SELECT archive_id, eso_category, access_url
    FROM phase3v2.product_files
    WHERE product_id = '{row['dp_id']}'
    ''').to_table()

    # 2. Filtrar el .tar
    es_tar = [str(c) == 'ANCILLARY.HARPSTAR' for c in archivos_harps['eso_category']]
    fila_tar = archivos_harps[es_tar][0]

    # 3. Descargar (la función ya se encarga de ver si ya existe)
    descargar(fila_tar['access_url'],
              Path('datos_LHS_1140_TAR') / 'HARPS' / f"{fila_tar['archive_id']}.tar")

Ya existe, no lo bajo otra vez: ADP.2015-11-24T02:01:17.500.tar
Ya existe, no lo bajo otra vez: ADP.2015-11-24T02:01:17.327.tar
Ya existe, no lo bajo otra vez: ADP.2015-11-28T02:00:55.867.tar
Ya existe, no lo bajo otra vez: ADP.2015-11-28T02:00:56.770.tar
Ya existe, no lo bajo otra vez: ADP.2015-11-29T02:00:53.053.tar
Ya existe, no lo bajo otra vez: ADP.2015-11-29T02:00:53.517.tar
Ya existe, no lo bajo otra vez: ADP.2015-11-30T02:00:55.203.tar
Ya existe, no lo bajo otra vez: ADP.2015-11-30T02:00:55.993.tar
Ya existe, no lo bajo otra vez: ADP.2015-12-01T02:01:28.467.tar
Ya existe, no lo bajo otra vez: ADP.2015-12-01T02:01:29.053.tar
Ya existe, no lo bajo otra vez: ADP.2015-12-02T02:00:53.943.tar
Ya existe, no lo bajo otra vez: ADP.2015-12-02T02:00:53.197.tar
Ya existe, no lo bajo otra vez: ADP.2015-12-09T10:14:36.283.tar
Ya existe, no lo bajo otra vez: ADP.2015-12-09T10:14:36.037.tar
Ya existe, no lo bajo otra vez: ADP.2015-12-10T02:01:24.740.tar
Ya existe, no lo bajo otra vez: ADP.2015

## Paso: Sacar el **e2ds_A** de cada **.tar**

In [ ]:
# Directorio donde se guarda los .tar en el Paso 1
directorio_tar = Path('datos_LHS_1140_TAR') / 'HARPS'

# Recorremos todos los .tar que descargaste
for ruta_tar in directorio_tar.glob('*.tar'):
    with tarfile.open(ruta_tar) as caja:
        # 1. Buscar el miembro e2ds_A dentro de la caja
        miembros_e2ds = [x for x in caja.getmembers() if x.name.endswith('e2ds_A.fits')]

        if not miembros_e2ds:
            print(f"No se encontró e2ds_A en: {ruta_tar.name}")
            continue

        miembro = miembros_e2ds[0]

        # 2. Extraer los datos en memoria bytes
        datos_bytes = caja.extractfile(miembro).read()

        # 3. Abrirlo con astropy usando BytesIO
        with fits_open(io.BytesIO(datos_bytes)) as hdul:
            header = hdul[0].header
            data = hdul[0].data  # Aquí está tu matriz (4096, 72)

            # Ejemplo de lo que se puede extraer de aquí para el siguiente paso:
            # fecha_obs = header.get('DATE-OBS')
            # blaze_file = header.get('ESO DRS BLAZE FILE')

            print(f"Procesado con éxito: {miembro.name} - Dimensiones: {data.shape}")

Procesado con éxito: data/reduced/2017-07-21/HARPS.2017-07-22T09:06:36.824_e2ds_A.fits - Dimensiones: (72, 4096)
Procesado con éxito: data/reduced/2017-11-18/HARPS.2017-11-19T04:24:21.371_e2ds_A.fits - Dimensiones: (72, 4096)
Procesado con éxito: data/reduced/2017-07-20/HARPS.2017-07-21T09:03:46.408_e2ds_A.fits - Dimensiones: (72, 4096)
Procesado con éxito: data/reduced/2015-11-22/HARPS.2015-11-23T03:30:42.912_e2ds_A.fits - Dimensiones: (72, 4096)
Procesado con éxito: data/reduced/2016-06-30/HARPS.2016-07-01T08:35:45.593_e2ds_A.fits - Dimensiones: (72, 4096)
Procesado con éxito: data/reduced/2016-08-06/HARPS.2016-08-07T09:42:47.692_e2ds_A.fits - Dimensiones: (72, 4096)
Procesado con éxito: data/reduced/2017-10-16/HARPS.2017-10-17T01:49:48.583_e2ds_A.fits - Dimensiones: (72, 4096)
Procesado con éxito: data/reduced/2016-10-19/HARPS.2016-10-20T03:09:08.525_e2ds_A.fits - Dimensiones: (72, 4096)
Procesado con éxito: data/reduced/2016-10-26/HARPS.2016-10-27T05:12:35.120_e2ds_A.fits - Dimensi

## Anotar qué blaze usó cada espectro

In [ ]:
# Ruta donde se guardan los archivos .tar
directorio_tar = Path('datos_LHS_1140_TAR') / 'HARPS'

# Recorrer todos los .tar descargados
for ruta_tar in directorio_tar.glob('*.tar'):
    with tarfile.open(ruta_tar) as caja:
        # Buscar el archivo e2ds_A
        miembros_e2ds = [x for x in caja.getmembers() if x.name.endswith('e2ds_A.fits')]
        if not miembros_e2ds:
            continue

        miembro = miembros_e2ds[0]
        datos = caja.extractfile(miembro).read()

    # Abrir en memoria para leer el header de cada uno
    with fits.open(io.BytesIO(datos)) as hdul:
        h = hdul[0].header
        print(f"--- {miembro.name} ---")
        for clave in h:
            if any(s in clave.upper() for s in ['BLAZE', 'FLAT FILE', 'TH FILE']):
                print(f"  {clave} = {h[clave]}")
    print()

--- data/reduced/2017-07-21/HARPS.2017-07-22T09:06:36.824_e2ds_A.fits ---
  ESO DRS CAL FLAT FILE = HARPS.2017-07-21T19:45:09.859_flat_A.fits
  ESO DRS BLAZE FILE = HARPS.2017-07-21T19:45:09.859_blaze_A.fits
  ESO DRS CAL TH FILE = HARPS.2017-07-21T19:59:08.916_wave_A.fits
  ESO DRS CAL TH FILE LINES = line_list_HARPS_A.dat

--- data/reduced/2017-11-18/HARPS.2017-11-19T04:24:21.371_e2ds_A.fits ---
  ESO DRS CAL FLAT FILE = HARPS.2017-11-18T20:52:26.133_flat_A.fits
  ESO DRS BLAZE FILE = HARPS.2017-11-18T20:52:26.133_blaze_A.fits
  ESO DRS CAL TH FILE = HARPS.2017-11-18T21:16:06.723_wave_A.fits
  ESO DRS CAL TH FILE LINES = line_list_HARPS_A.dat

--- data/reduced/2017-07-20/HARPS.2017-07-21T09:03:46.408_e2ds_A.fits ---
  ESO DRS CAL FLAT FILE = HARPS.2017-07-20T19:05:15.800_flat_A.fits
  ESO DRS BLAZE FILE = HARPS.2017-07-20T19:05:15.800_blaze_A.fits
  ESO DRS CAL TH FILE = HARPS.2017-07-20T19:22:18.836_wave_A.fits
  ESO DRS CAL TH FILE LINES = line_list_HARPS_A.dat

--- data/reduced/20